# Option 1 (P4) — training-configuration inspection

Candidate training config for the window-size sweep, evaluated on JHH369 at config A
geometry (FOV 128 µm, 2.0 µm/px).

**Training config**
`n_crops_for_tissue_train 1024 | weight_decay 0.0 | n_element_min_for_crop 10 | warm_down_epochs 100`

**Analysis is identical across all three options**: tiling, no overlap, seed 0, and a
50-cell window filter — so any difference here comes from *training* alone. Note the
deliberate asymmetry in options 1 and 3: threshold 10 at training, 50 at analysis. The
border problem is an analysis artefact, so it is fixed at analysis time.

Companions: `S1b` (opt 1), `S1c` (opt 2), `S1d` (opt 3).

In [ ]:
%matplotlib inline
import sys
from pathlib import Path
# resolve imc_tm from anywhere: repo root, tissuemosaic/, notebooks/, notebooks/archive/
_cands = [c for p in [Path.cwd(), *Path.cwd().parents] for c in (p, p / "tissuemosaic")]
sys.path.insert(0, str(next(c for c in _cands if (c / "imc_tm.py").exists())))
import imc_tm
import numpy as np, pandas as pd, matplotlib.pyplot as plt, seaborn
import warnings; warnings.filterwarnings("ignore")
pd.set_option("display.width", 250)

LABEL = "Option 1 (P4)"
CKPT  = Path(r"/tmp/claude-1000/-home-gavehan-PDACAntigenPresentationABMs/998c13c4-6095-455b-818b-07c1eee15128/scratchpad/probes2/P4_thr10_wd0_warmdown100/ckpt_last.pt")
ANALYSIS_THR, RESOLUTION, N_NEIGHBORS, SEED = 50, 1.0, 15, 0

print(LABEL); print("n_crops_for_tissue_train 1024 | weight_decay 0.0 | n_element_min_for_crop 10 | warm_down_epochs 100")
print("checkpoint:", CKPT, "| exists:", CKPT.is_file())
print(imc_tm.compat_summary())

## Training curve

In [ ]:
import re
log = CKPT.parent/"train.log"
txt = log.read_text(errors="ignore").replace("\r","\n") if log.is_file() else ""
per = {int(e): float(l) for e,l in re.findall(r"Epoch (\d+):.*?loss=([0-9.]+)", txt)}
curve = pd.DataFrame(sorted(per.items()), columns=["epoch","loss"])
import yaml
cfg_p = CKPT.parent/"config.yaml"
cfg = yaml.safe_load(open(cfg_p)) if cfg_p.is_file() else imc_tm.load_config()
fig, ax = plt.subplots(figsize=(8,3.6))
ax.plot(curve.epoch, curve.loss, lw=1.4)
ax.axvspan(0, cfg["warm_up_epochs"], color="C1", alpha=.12, label="LR warm-up")
ax.axvspan(cfg["max_epochs"]-cfg["warm_down_epochs"], cfg["max_epochs"], color="C2", alpha=.12, label="LR cosine decay")
ax.set_xlabel("epoch"); ax.set_ylabel("DINO loss"); ax.set_title(LABEL); ax.legend(fontsize=8)
plt.tight_layout()
print(f"{len(curve)} epochs | {curve.loss.iloc[0]:.3f} -> {curve.loss.iloc[-1]:.3f}"
      f" | loss@89 = {per.get(89)}")
print("reference:  v1 final 0.756 (loss@89 1.36)   v2 final 2.82 (loss@89 2.78)")

## Featurise and cluster

In [ ]:
adatas = {r:a for r,a in imc_tm.load_all_anndata().items() if r.startswith(imc_tm.SWEEP_PATIENT)}
acfg = imc_tm.sweep_config_dict(imc_tm.sweep_spec("A_fov128_px2.0"))
acfg["n_element_min_for_crop"] = ANALYSIS_THR
model = imc_tm.load_model(CKPT); dm = imc_tm.make_datamodule(acfg)
res = imc_tm.featurize_windows(model, dm, adatas, frac_overlap=0.0, seed=SEED, verbose=True)
del model
cl  = imc_tm.cluster_embedding(res["features"], n_neighbors=N_NEIGHBORS, resolution=RESOLUTION, seed=SEED)
lab = cl["labels"]
met = imc_tm.window_metrics(res, lab, k=4, n_perm=200, seed=SEED)
print()
for k,v in met.items(): print(f"  {k:<24s} {v:.4f}" if isinstance(v,float) else f"  {k:<24s} {v}")

## Border-cluster diagnostic

The failure seen in v1: two of nine clusters were **100%** and **86%** border windows —
windows sitting partly outside the tissue. A border ring is contiguous by construction,
so it inflates spatial coherence, and its share scales with FOV. The 50-cell analysis
filter is meant to remove them; this checks whether it did.

In [ ]:
bdf, bsum = imc_tm.border_cluster_report(res["centers_um"], res["roi"], lab)
print("overall border windows: %.1f%%" % bsum["overall_border_pct"])
print("clusters >=80%% border : %d   (v1 unfiltered had 2)" % bsum["n_border_clusters_ge80pct"])
print("worst cluster          : %.1f%% border" % bsum["max_cluster_border_pct"])
print()
display(bdf.style.background_gradient(subset=["pct_border"], cmap="Reds", vmin=0, vmax=100))

## UMAP

In [ ]:
CLUSTER_COLORS = imc_tm.cluster_palette(labels=sorted(np.unique(lab)))
ROIS = sorted(adatas)
ROI_COLORS = imc_tm.roi_palette(ROIS)
fig, axes = plt.subplots(1,2, figsize=(15,6.2))
for c in sorted(np.unique(lab)):
    m = lab==c
    axes[0].scatter(cl["umap"][m,0], cl["umap"][m,1], color=CLUSTER_COLORS[c], s=26, linewidths=0,
                    label=f"cluster {c}  (n={m.sum()})")
axes[0].set_title(f"Leiden cluster — {met['n_clusters']} clusters")
axes[0].legend(bbox_to_anchor=(1.01,1), loc="upper left", fontsize=8, frameon=False)
for r in ROIS:
    m = res["roi"]==r
    axes[1].scatter(cl["umap"][m,0], cl["umap"][m,1], color=ROI_COLORS[r], s=26, linewidths=0,
                    label=f"{r}  (n={m.sum()})")
axes[1].set_title(f"ROI — roi_mixing {met['roi_mixing']:.3f} (null ~0.97)")
axes[1].legend(bbox_to_anchor=(1.01,1), loc="upper left", fontsize=8, frameon=False)
for a in axes: a.set_xticks([]); a.set_yticks([]); a.set_xlabel("UMAP-1"); a.set_ylabel("UMAP-2")
plt.suptitle(f"{LABEL} — {len(lab)} windows", y=1.01, fontsize=13); plt.tight_layout()

## Clusters in tissue

In [ ]:
fig, axes = plt.subplots(1, len(ROIS), figsize=(5.0*len(ROIS), 5.4))
axes = np.atleast_1d(axes)
for ax, r in zip(axes, ROIS):
    m = res["roi"]==r
    ax.scatter(res["centers_um"][m,0], res["centers_um"][m,1],
               color=[CLUSTER_COLORS[c] for c in lab[m]], s=250, marker="s", linewidths=0)
    ax.set_aspect("equal"); ax.set_xticks([]); ax.set_yticks([]); ax.set_title(f"{r}  ({m.sum()})")
handles = [plt.Line2D([0],[0],marker="s",linestyle="",markersize=9,color=CLUSTER_COLORS[c],label=f"cluster {c}")
           for c in sorted(np.unique(lab))]
fig.legend(handles=handles, bbox_to_anchor=(1.005,0.5), loc="center left", fontsize=9, frameon=False)
plt.suptitle(f"{LABEL} — spatial_coherence z = {met['spatial_coherence_z']:.1f}", y=1.02, fontsize=13)
plt.tight_layout()

## Composition per cluster

In [ ]:
comp = pd.DataFrame(res["composition"], columns=imc_tm.CELL_TYPES); comp["cluster"] = lab
prof = comp.groupby("cluster")[imc_tm.CELL_TYPES].mean()
prof.insert(0,"n_windows", comp.groupby("cluster").size())
prof.loc["ALL"] = [len(lab)] + list(res["composition"].mean(axis=0))
display(prof.round(3).style.background_gradient(axis=0, cmap="Blues", subset=imc_tm.CELL_TYPES))

In [ ]:
cell_colors = imc_tm.palette()
pb = prof.drop(index="ALL")[imc_tm.CELL_TYPES]; pb.loc["ALL"] = res["composition"].mean(axis=0)
bio = pb[imc_tm.BIOLOGY_TYPES]; bio = bio.div(bio.sum(axis=1), axis=0)
fig, axes = plt.subplots(1,2, figsize=(17,5.4))
for ax,(tbl,cols,title) in zip(axes, [(pb, imc_tm.CELL_TYPES, "all 10 channels"),
                                      (bio, imc_tm.BIOLOGY_TYPES, "classified biology only, renormalised")]):
    bottom = np.zeros(len(tbl)); x = np.arange(len(tbl))
    for t in cols:
        ax.bar(x, tbl[t].values, bottom=bottom, color=cell_colors[t], label=t, width=0.82); bottom += tbl[t].values
    ax.set_xticks(x); ax.set_xticklabels([f"c{i}" if i!="ALL" else "ALL" for i in tbl.index], fontsize=9)
    ax.set_xlabel("cluster"); ax.set_ylabel("mean fraction"); ax.set_ylim(0,1); ax.set_title(title)
    ax.axvline(len(tbl)-1.5, color="0.3", lw=1, ls=":")
handles = [plt.Rectangle((0,0),1,1,color=cell_colors[t],
           label=t + ("  (background)" if t in imc_tm.BACKGROUND_TYPES else "")) for t in imc_tm.CELL_TYPES]
fig.legend(handles=handles, bbox_to_anchor=(1.005,0.5), loc="center left", fontsize=9, frameon=False, title="cell type")
plt.suptitle(f"{LABEL} — composition per cluster", y=1.02, fontsize=13); plt.tight_layout()

In [ ]:
ct = pd.crosstab(lab, res["roi"]); ct["total"] = ct.sum(axis=1)
print(ct.to_string())